### LCEL Deepdive

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
# from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from euriai.langchain import create_chat_model
import time

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key")

#chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)

### API using openrouter ----
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(
    model="stepfun/step-3.5-flash:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("openrouter_api"),
    # model_kwargs={
    #     "reasoning": {
    #         "enabled": True
    #     }
    # }
)

response = chat_model.invoke("What is the meaning of life?")
print(response.content)

That is perhaps humanity's oldest and most profound question. There is no single, universally accepted answer—instead, different perspectives offer various interpretations. Here’s a brief overview of how thinkers, cultures, and disciplines approach it:

---

### **1. Philosophical Perspectives**
- **Existentialism** (e.g., Sartre, Camus):  
  Life has no inherent meaning—we create our own through choices, actions, and commitments. “Existence precedes essence.” Meaning comes from living authentically, even in an indifferent universe.
- **Absurdism** (Camus):  
  Humans seek meaning in a meaningless universe. The “absurd” arises from this clash—we can either despair or revolt by embracing life despite its lack of inherent purpose.
- **Stoicism** (Marcus Aurelius, Seneca):  
  Meaning lies in living virtuously, accepting what we cannot control, and contributing to the common good with wisdom and courage.
- **Utilitarianism & altruism**:  
  Meaning comes from maximizing happiness or reduc

In [12]:
prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic}")
model = chat_model
output_parser = StrOutputParser()

chain = prompt | model | output_parser

chain.invoke({"topic": "ice cream"})

'Why did the ice cream go to the party? Because it was feeling a little "chill"ed!'

In [13]:
prompt.invoke({"topic": "ice cream"})

ChatPromptValue(messages=[HumanMessage(content='tell me a short joke about ice cream', additional_kwargs={}, response_metadata={})])

In [14]:
from langchain_core.messages.human import HumanMessage

messages = [HumanMessage(content='tell me a short joke about ice cream')]
model.invoke(messages)

AIMessage(content='Why did the ice cream go to the party? Because it was cool with everyone!', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 17, 'completion_tokens': 17, 'total_tokens': 34}, 'model_name': 'gpt-4.1-nano', 'system_fingerprint': None, 'finish_reason': 'stop', 'model': 'gpt-4.1-nano', 'created': 1773132919}, id='lc_run--019cd6f5-30c2-76a0-886e-da3887d2f094-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 17, 'total_tokens': 34})

### Why use "|" in langchain ?


In [ ]:
from abc import ABC, abstractmethod

class CRunnable(ABC):
    def __init__(self):
        self.next = None
    
    @abstractmethod
    def process(self, data):
        pass

    def invoke(self, data):
        processed_data = self.process(data)
        if self.next is not None:
            return self.next.invoke(processed_data)
        return processed_data

In [11]:
x = CRunnable()
x.next

1

In [4]:
from abc import ABC, abstractmethod

class CRunnable(ABC):
    def __init__(self):
        self.next = None

    @abstractmethod
    def process(self, data):
        """
        This method must be implemented by subclasses to define
        data processing behavior.
        """
        pass

    def invoke(self, data):
        processed_data = self.process(data)
        if self.next is not None:
            return self.next.invoke(processed_data)
        return processed_data

    def __or__(self, other):
        return CRunnableSequence(self, other)

class CRunnableSequence(CRunnable):
    def __init__(self, first, second):
        super().__init__()
        self.first = first
        self.second = second

    def process(self, data):
        return data

    def invoke(self, data):
        first_result = self.first.invoke(data)
        return self.second.invoke(first_result)

In [5]:
class AddTen(CRunnable):
    def process(self, data):
        print("AddTen: ", data)
        return data + 10

class MultiplyByTwo(CRunnable):
    def process(self, data):
        print("Multiply by 2: ", data)
        return data * 2

class ConvertToString(CRunnable):
    def process(self, data):
        print("Convert to string: ", data)
        return f"Result: {data}"

In [6]:
a = AddTen()
b = MultiplyByTwo()
c = ConvertToString()

chain = a | b | c

result = chain.invoke(10)
print(result)

AddTen:  10
Multiply by 2:  20
Convert to string:  40
Result: 40


### Runnables from LangChain

In [15]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

In [16]:
chain = RunnablePassthrough()
chain.invoke("hello")

'hello'

In [17]:
chain = RunnablePassthrough() | RunnablePassthrough () | RunnablePassthrough ()
chain.invoke("hello")

'hello'

In [18]:
def input_to_upper(input: str):
    output = input.upper()
    return output

In [19]:
chain = RunnableLambda(input_to_upper)
chain.invoke("hello")

'HELLO'

In [20]:
chain = RunnablePassthrough() | RunnableLambda(input_to_upper) | RunnablePassthrough()
chain.invoke("hello")

'HELLO'

In [23]:
chain = RunnableParallel(
    {
        "x": RunnablePassthrough(), 
        "y": RunnablePassthrough()
    }
)
print(chain.invoke("hello"))
print("-------------------------------------------------------")
print(chain.invoke({"input": "hello", "input2": "goodbye"}))

{'x': 'hello', 'y': 'hello'}
-------------------------------------------------------
{'x': {'input': 'hello', 'input2': 'goodbye'}, 'y': {'input': 'hello', 'input2': 'goodbye'}}


In [25]:
chain = RunnableParallel(
    {
        "x": RunnablePassthrough(), 
        "y": RunnableLambda(lambda z: z["input2"])
    }
)

chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'}, 'y': 'goodbye'}

### Nested chains - now it gets more complicated!

In [26]:
def find_keys_to_uppercase(input: dict):
    output = input.get("input", "not found").upper()
    return output

chain = RunnableParallel(
    {"x": RunnablePassthrough() | RunnableLambda(find_keys_to_uppercase), 
     "y": RunnableLambda(lambda z: z["input2"])})

chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': 'HELLO', 'y': 'goodbye'}

In [28]:
chain = RunnableParallel(
    {
        "x": RunnablePassthrough()
    }
)

chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'}}

In [29]:
def assign_func(_):
    return 100

def multiply(input):
    return input * 10

chain = RunnableParallel(
    {
        "x": RunnablePassthrough()}).assign(extra=RunnableLambda(assign_func))

result = chain.invoke({"input": "hello", "input2": "goodbye"})
print(result)

{'x': {'input': 'hello', 'input2': 'goodbye'}, 'extra': 100}


### Combine multiple chains

In [30]:
def extractor(input: dict):
    return input.get("extra", "Key not found")

def cupper(upper: str):
    return str(upper).upper()

new_chain = RunnableLambda(extractor) | RunnableLambda(cupper)

new_chain.invoke({"extra": "test"})

'TEST'

In [31]:
final_chain = chain | new_chain
final_chain.invoke({"input": "hello", "input2": "goodbye"})

'100'